# SPX RTH 原始路径 × 期权结构事件研究

## tl;dr

- direction_up_5m: combined−path Brier +0.0059, session bootstrap 95% CI [-0.0008, +0.0136]，未确认。
- breakout_followthrough_15m: combined−path Brier -0.0054, session bootstrap 95% CI [-0.0126, +0.0024]，未确认。
- reversal_15m: combined−path Brier +0.0020, session bootstrap 95% CI [-0.0031, +0.0077]，未确认。
- pullback_resume_15m: combined−path Brier +0.0011, session bootstrap 95% CI [-0.0006, +0.0031]，未确认。
- latent direction_up_5m: state+path−path Brier +0.0008, 95% CI [-0.0005, +0.0021]，状态增量未确认。
- latent breakout_followthrough_15m: state+path−path Brier +0.0002, 95% CI [-0.0010, +0.0015]，状态增量未确认。
- latent reversal_15m: state+path−path Brier -0.0012, 95% CI [-0.0031, +0.0011]，状态增量未确认。
- latent pullback_resume_15m: state+path−path Brier +0.0003, 95% CI [-0.0001, +0.0006]，状态增量未确认。
- wall competing-risk: walls−path multiclass Brier -0.0133, 95% CI [-0.0234, -0.0027]。
- change-point reversal reversal_15m: ΔBrier +0.0006, 95% CI [-0.0004, +0.0016]。
- change-point reversal reversal_15m: ΔBrier +0.0019, 95% CI [-0.0028, +0.0073]。
- 5s motif direction_up_5m: ΔBrier +0.0008, 95% CI [-0.0005, +0.0021]。
- 5s motif breakout_followthrough_15m: ΔBrier +0.0013, 95% CI [+0.0001, +0.0025]。
- 5s motif reversal_15m: ΔBrier +0.0006, 95% CI [-0.0011, +0.0024]。
- 5s motif pullback_resume_15m: ΔBrier +0.0000, 95% CI [-0.0003, +0.0004]。
- sparse MoE breakout_followthrough_15m: ΔBrier +0.0003, 95% CI [-0.0019, +0.0027]。
- sparse MoE reversal_15m: ΔBrier +0.0002, 95% CI [-0.0013, +0.0018]。
- sparse MoE pullback_resume_15m: ΔBrier +0.0002, 95% CI [-0.0001, +0.0006]。

## Context & Methods

- 不读取 production strategy/candidate 作为样本或标签。
- 结构快照必须先于 decision time；原始 SPX/VIX/VIX1D 仅作 backward as-of join。
- 标签覆盖 5m direction、15m 墙位突破延续、反转、回撤恢复。
- 突破额外拆成上破站稳 / 下破站稳 / 未突破 competing risk。
- 5 秒路径构造因果 CUSUM/BOCPD 与 fold-local motif dictionary。
- sparse MoE gate 固定只看 15m return 与 Call/Put wall 距离。
- 以前 10 个 session 起步，之后 expanding session-held-out walk-forward。
- 主比较为 path-only 与 path+structure 的 Brier 差，按 session bootstrap。

### Key Assumptions

GEX 采用 Call+ / Put− OI proxy，不代表真实 dealer 仓位；结构用于条件化，不能直接宣称对冲流方向。

## Data

读取 5 分钟 IV surface、点时 Greeks/OI proxy 和 Schwab live SPX/VIX/VIX1D。

In [1]:
import importlib.util
import json
spec = importlib.util.spec_from_file_location("structure_event_study", '/home/ubuntu/spx-spark/docs/notebooks/strategy_structure_event_study_2026_08_19.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
analysis = module.run_analysis()
print(json.dumps(analysis["quality"], indent=2, ensure_ascii=False))

{
  "surface_rows": 1861,
  "model_rows": 1794,
  "sessions": 27,
  "first_session": "2026-07-13",
  "last_session": "2026-08-18",
  "skipped": {
    "path_incomplete": 67,
    "five_second_path_incomplete": 13
  }
}


## Results

In [2]:
for target in module.TARGETS:
    print("\n", target)
    for row in module.metric_rows(analysis, target):
        print(row)
print("\nStructure increment")
for row in analysis["structure_increment"]:
    print(row)
print("\nBest one-factor increments by target")
for target in module.TARGETS:
    rows = sorted(
        (row for row in analysis["factor_ablation"] if row["target"] == target),
        key=lambda row: row["delta_brier"],
    )
    print(target, rows[:5])
print("\nLatent-state OOS increment")
for row in analysis["latent_state_increment"]:
    print(row)
print("\nDescriptive state space")
print(json.dumps(analysis["descriptive_state_space"], indent=2, ensure_ascii=False))
print("\nWall competing-risk hazard")
for row in analysis["competing_risk_metrics"]:
    print(row)
print("increment", analysis["competing_risk_increment"])
print("\nCUSUM/BOCPD reversal")
for row in analysis["change_point_metrics"]:
    print(row)
print("increment", analysis["change_point_increment"])
print("\n5-second motif dictionary")
for row in analysis["motif_increment"]:
    print(row)
print("\nSparse two-expert model")
for row in analysis["sparse_moe_increment"]:
    print(row)


 direction_up_5m
{'target': 'direction_up_5m', 'model': 'intercept', 'n': 1099, 'sessions': 17, 'base_rate': 0.5177434030937216, 'brier': 0.2510108817642399, 'log_loss': 0.6951704817390845, 'auc': 0.47458964751135724}
{'target': 'direction_up_5m', 'model': 'path', 'n': 1099, 'sessions': 17, 'base_rate': 0.5177434030937216, 'brier': 0.2517441776585747, 'log_loss': 0.6966706204139903, 'auc': 0.49601750837284875}
{'target': 'direction_up_5m', 'model': 'structure', 'n': 1099, 'sessions': 17, 'base_rate': 0.5177434030937216, 'brier': 0.2564285303433629, 'log_loss': 0.7072926632688926, 'auc': 0.5003912856053321}
{'target': 'direction_up_5m', 'model': 'path+structure', 'n': 1099, 'sessions': 17, 'base_rate': 0.5177434030937216, 'brier': 0.2574331313924821, 'log_loss': 0.7096468467061188, 'auc': 0.507421162582485}

 breakout_followthrough_15m
{'target': 'breakout_followthrough_15m', 'model': 'intercept', 'n': 1050, 'sessions': 17, 'base_rate': 0.24095238095238095, 'brier': 0.18314263288305596

In [3]:
print(json.dumps(analysis['feature_coverage'], indent=2, ensure_ascii=False))

[
  {
    "feature": "return_1m_scale",
    "available_rate": 1.0
  },
  {
    "feature": "return_5m_scale",
    "available_rate": 1.0
  },
  {
    "feature": "return_15m_scale",
    "available_rate": 1.0
  },
  {
    "feature": "range_15m_scale",
    "available_rate": 1.0
  },
  {
    "feature": "range_position_15m",
    "available_rate": 1.0
  },
  {
    "feature": "zero_gamma_distance_scale",
    "available_rate": 0.5763656633221851
  },
  {
    "feature": "call_wall_distance_scale",
    "available_rate": 0.9604236343366778
  },
  {
    "feature": "put_wall_distance_scale",
    "available_rate": 0.9604236343366778
  },
  {
    "feature": "expected_move_scale",
    "available_rate": 0.9537346711259754
  },
  {
    "feature": "atm_iv",
    "available_rate": 0.9604236343366778
  },
  {
    "feature": "atm_iv_jump_5m",
    "available_rate": 0.9370122630992196
  },
  {
    "feature": "put_skew_25d",
    "available_rate": 0.9604236343366778
  },
  {
    "feature": "put_skew_change_5m",
  

## Takeaways

- direction_up_5m: combined−path Brier +0.0059, session bootstrap 95% CI [-0.0008, +0.0136]，未确认。
- breakout_followthrough_15m: combined−path Brier -0.0054, session bootstrap 95% CI [-0.0126, +0.0024]，未确认。
- reversal_15m: combined−path Brier +0.0020, session bootstrap 95% CI [-0.0031, +0.0077]，未确认。
- pullback_resume_15m: combined−path Brier +0.0011, session bootstrap 95% CI [-0.0006, +0.0031]，未确认。
- latent direction_up_5m: state+path−path Brier +0.0008, 95% CI [-0.0005, +0.0021]，状态增量未确认。
- latent breakout_followthrough_15m: state+path−path Brier +0.0002, 95% CI [-0.0010, +0.0015]，状态增量未确认。
- latent reversal_15m: state+path−path Brier -0.0012, 95% CI [-0.0031, +0.0011]，状态增量未确认。
- latent pullback_resume_15m: state+path−path Brier +0.0003, 95% CI [-0.0001, +0.0006]，状态增量未确认。
- wall competing-risk: walls−path multiclass Brier -0.0133, 95% CI [-0.0234, -0.0027]。
- change-point reversal reversal_15m: ΔBrier +0.0006, 95% CI [-0.0004, +0.0016]。
- change-point reversal reversal_15m: ΔBrier +0.0019, 95% CI [-0.0028, +0.0073]。
- 5s motif direction_up_5m: ΔBrier +0.0008, 95% CI [-0.0005, +0.0021]。
- 5s motif breakout_followthrough_15m: ΔBrier +0.0013, 95% CI [+0.0001, +0.0025]。
- 5s motif reversal_15m: ΔBrier +0.0006, 95% CI [-0.0011, +0.0024]。
- 5s motif pullback_resume_15m: ΔBrier +0.0000, 95% CI [-0.0003, +0.0004]。
- sparse MoE breakout_followthrough_15m: ΔBrier +0.0003, 95% CI [-0.0019, +0.0027]。
- sparse MoE reversal_15m: ΔBrier +0.0002, 95% CI [-0.0013, +0.0018]。
- sparse MoE pullback_resume_15m: ΔBrier +0.0002, 95% CI [-0.0001, +0.0006]。

任何区间跨 0 的结果都只算未确认；下一步只能增加独立 session，不能靠继续调阈值制造显著性。

In [4]:
assert analysis["quality"]["sessions"] >= 15
assert all(row["sessions"] >= 5 for row in analysis["structure_increment"])
assert all(row["sessions"] >= 5 for row in analysis["latent_state_increment"])
assert analysis["competing_risk_increment"]["sessions"] >= 5
assert all(row["sessions"] >= 5 for row in analysis["change_point_increment"])
assert all(row["sessions"] >= 5 for row in analysis["motif_increment"])
assert all(row["sessions"] >= 5 for row in analysis["sparse_moe_increment"])
assert 2 <= analysis["descriptive_state_space"]["selected_components"] <= 6
assert all(
    row["available_rate"] >= 0.50
    for row in analysis["feature_coverage"]
    if row["feature"] in analysis["feature_sets"]["path+structure"]
)
print("causal/session-held-out acceptance checks passed")

causal/session-held-out acceptance checks passed
